## Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

## Folder

In [ ]:
folder = f"../results/run_{random.randint(100000, 999999)}"
os.makedirs(folder, exist_ok=True)

## Parameters

### Inputs

In [ ]:
radnet_input_df = pd.read_csv("../inputs/radnet_input.csv")
radnet_input_class = radnet_input_df['label'].values
radnet_input_efsr = radnet_input_df['efsr_edge'].values

### Outputs

In [ ]:
inference_output_df = pd.read_csv("../outputs/inference_output.csv")
inference_output_class = inference_output_df['class'].values
inference_output_confidence = inference_output_df['confidence'].values

In [ ]:
fuzzy_output_df = pd.read_csv("../outputs/fuzzy_output.csv")
fuzzy_output_class = fuzzy_output_df['class'].values
fuzzy_output_confidence = fuzzy_output_df['confidence'].values

## Confiança

### Confiança x Arestas

In [ ]:
plt.figure(figsize=(10,6))

x_values = np.arange(1, len(fuzzy_output_confidence) + 1)

plt.plot(x_values, fuzzy_output_confidence, label="Fuzzy Logic", alpha=0.7)

plt.title("Confidence: Fuzzy Logic", fontsize=12)
plt.xlabel("Edges", fontsize=12)
plt.ylabel("Confidence", fontsize=12)

plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.5, 1)

n = len(fuzzy_output_confidence)
ticks = [1] + list(range(5, n + 1, 5))
plt.xticks(ticks)

plt.savefig(f'{folder}/fuzzy_logic.png', bbox_inches='tight')
plt.show()

## Classificação

### Lógica Fuzzy

In [ ]:
plt.figure(figsize=(12, 6))

x_values = np.arange(1, len(fuzzy_output_class) + 1)

conf_mantida = fuzzy_output_confidence.copy()
conf_invertida = np.zeros(len(fuzzy_output_confidence))

indices_invertidos = np.where(fuzzy_output_class != inference_output_class)[0]

conf_mantida[indices_invertidos] = 0

for i in indices_invertidos:
    conf_invertida[i] = -fuzzy_output_confidence[i]

plt.bar(x_values, conf_mantida, color='#4472C4', label='Classification maintained', alpha=0.9)
plt.bar(x_values, conf_invertida, color='#ED7D31', label='Classification inverted', alpha=0.9)

plt.title("Confidence Fuzzy: Maintained vs Inverted", fontsize=12)
plt.xlabel("Edges", fontsize=12)
plt.ylabel("Confidence", fontsize=12)
plt.ylim(-1.1, 1.1)
plt.axhline(y=0, color='black', linewidth=1.2, linestyle='-')
plt.axhline(y=0.5, color='green', linewidth=0.8, linestyle='--', alpha=0.5, label='Limiar (0.5)')
plt.axhline(y=-0.5, color='green', linewidth=0.8, linestyle='--', alpha=0.5)
plt.legend(fontsize=10, loc='upper right')
plt.grid(True, alpha=0.3, axis='y')

n = len(fuzzy_output_class)
ticks = [1] + list(range(5, n + 1, 5))
plt.xticks(ticks)

y_ticks = [-1.0, -0.5, 0, 0.5, 1.0]
y_labels = ['1.0', '0.5', '0', '0.5', '1.0']
plt.yticks(y_ticks, y_labels)

plt.savefig(f'{folder}/fuzzy_confidence_inverted.png', bbox_inches='tight', dpi=300)
plt.show()

## Matrizes de Confusão

### Ground Truth

In [ ]:
def create_ground_truth(class_a, class_b, confidence_a, confidence_b, threshold=0.5):
    ground_truth = []
    
    for i in range(len(class_a)):
        score_a = confidence_a[i] if class_a[i] == 1 else (1 - confidence_a[i])
        score_b = confidence_b[i] if class_b[i] == 1 else (1 - confidence_b[i])
        
        score = (score_a + score_b) / 2
        
        if score > threshold:
            ground_truth.append(1)
        else:
            ground_truth.append(0)
    
    return np.array(ground_truth)

In [ ]:
# Normalização
radnet_input_confidence = np.clip((radnet_input_efsr - 0.20) / (0.98 - 0.20), 0, 1)

In [ ]:
ground_truth = create_ground_truth(
    radnet_input_class, 
    inference_output_class,
    radnet_input_confidence,
    inference_output_confidence
)

### Matriz de Confusão: Lógica Fuzzy

In [ ]:
acc = accuracy_score(ground_truth, fuzzy_output_class)
precision = precision_score(ground_truth, fuzzy_output_class)
recall = recall_score(ground_truth, fuzzy_output_class)
f1 = f1_score(ground_truth, fuzzy_output_class)

cm = confusion_matrix(ground_truth, fuzzy_output_class)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Bad (0)', 'Good (1)'], 
            yticklabels=['Bad (0)', 'Good (1)'])
plt.xlabel('Prediction', fontsize=12)
plt.ylabel('Ground Truth', fontsize=12)
plt.title(f'Confusion Matrix: Fuzzy Logic', fontsize=14)
plt.tight_layout()

plt.savefig(f'{folder}/confusion_matrix_fuzzy_logic.png', bbox_inches='tight')
plt.show()

## Métricas

In [ ]:
with open(f'{folder}/metrics.txt', 'w') as f:
    f.write("Metrics\n")
    f.write("-" * 25 + "\n")
    f.write(f"Accuracy:        {acc*100:.2f}%\n")
    f.write(f"Precision:       {precision*100:.2f}%\n")
    f.write(f"Recall:          {recall*100:.2f}%\n")
    f.write(f"F1:              {f1*100:.2f}%\n")

In [ ]:
print("Metrics")
print("-" * 25)
print(f"Accuracy:        {acc*100:.2f}%")
print(f"Precision:       {precision*100:.2f}%")
print(f"Recall:          {recall*100:.2f}%")
print(f"F1:              {f1*100:.2f}%")